In [ ]:
# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

# Test: Evaluation Runner

This notebook tests the `run_evaluation.py` script and its components:
- GoldenOutputManager (caching)
- Golden dataset loading
- Ticker processing
- Full evaluation pipeline
- Metrics integration

## Setup

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
from datetime import datetime

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path updated: {str(project_root) in sys.path}")

Project root: c:\Users\H244746\Documents\reit-risk-summarizer
Python path updated: True


In [3]:
from evaluation.golden_output_manager import GoldenOutputManager

print("✓ Successfully imported evaluation modules")

✓ Successfully imported evaluation modules


## Test 1: GoldenOutputManager

In [4]:
# Test 1.1: Initialize manager
print("Test 1.1: Initialize GoldenOutputManager\n")

golden_manager = GoldenOutputManager()
print(f"Cache directory: {golden_manager.cache_dir}")
print(f"Directory exists: {golden_manager.cache_dir.exists()}")
print(f"✓ PASS" if golden_manager.cache_dir.exists() else "✗ FAIL")

Test 1.1: Initialize GoldenOutputManager

Cache directory: c:\Users\H244746\Documents\reit-risk-summarizer\evaluation\golden_outputs
Directory exists: True
✓ PASS


In [6]:
# Test 1.2: List cached outputs
print("Test 1.2: List cached golden outputs\n")

cached_outputs = golden_manager.list_cached_tickers()
print(f"Found {len(cached_outputs)} cached outputs:")
for ticker in cached_outputs:
    print(f"  - {ticker}")

print(f"\n✓ PASS (found cached outputs)" if len(cached_outputs) > 0 else "⚠ No cached outputs found")

Test 1.2: List cached golden outputs

Found 3 cached outputs:
  - AMT
  - EQIX
  - PLD

✓ PASS (found cached outputs)


In [7]:
# Test 1.3: Load cached output
print("Test 1.3: Load cached output for a ticker\n")

if len(cached_outputs) > 0:
    test_ticker = cached_outputs[0]
    cached_summary = golden_manager.load_cached_output(test_ticker)
    
    if cached_summary:
        print(f"Loaded cached output for {test_ticker}:")
        print(f"  Company: {cached_summary.company_name}")
        print(f"  Sector: (check golden dataset)")
        print(f"  Model: {cached_summary.model}")
        print(f"  Prompt: {cached_summary.prompt_version}")
        print(f"  Risks: {len(cached_summary.risks)}")
        print(f"\nRisk 1: {cached_summary.risks[0][:100]}...")
        print(f"\n✓ PASS")
    else:
        print(f"✗ FAIL: Could not load cached output for {test_ticker}")
else:
    print("⚠ No cached outputs to test")

Test 1.3: Load cached output for a ticker

Loaded cached output for AMT:
  Company: American Tower
  Sector: (check golden dataset)
  Model: llama-3.3-70b-versatile
  Prompt: v1.0
  Risks: 5

Risk 1: Substantial revenue from small number of wireless carriers (AT&T, Verizon, T-Mobile)...

✓ PASS


In [8]:
# Test 1.4: Check cache path format
print("Test 1.4: Verify cache path format\n")

test_ticker = "TEST"
cache_path = golden_manager.get_cache_path(test_ticker)
print(f"Cache path for {test_ticker}: {cache_path}")
print(f"Expected filename: {test_ticker}.json")
print(f"\n✓ PASS" if cache_path.name == f"{test_ticker}.json" else "✗ FAIL")

Test 1.4: Verify cache path format

Cache path for TEST: c:\Users\H244746\Documents\reit-risk-summarizer\evaluation\golden_outputs\TEST.json
Expected filename: TEST.json

✓ PASS


## Test 2: Golden Dataset Loading

In [9]:
# Test 2.1: Load golden dataset CSV
print("Test 2.1: Load golden dataset\n")

dataset_path = project_root / "evaluation" / "golden_dataset.csv"
df = pd.read_csv(dataset_path)

print(f"Loaded {len(df)} rows from golden dataset")
print(f"Unique tickers: {df['ticker'].nunique()}")
print(f"Unique sectors: {df['sector'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\n✓ PASS")

Test 2.1: Load golden dataset

Loaded 50 rows from golden dataset
Unique tickers: 10
Unique sectors: 8

Columns: ['ticker', 'company_name', 'sector', 'filing_year', 'risk_rank', 'risk_category', 'risk_title', 'risk_description', 'why_material', 'unique_to_sector']

✓ PASS


In [10]:
# Test 2.2: Group risks by ticker
print("Test 2.2: Group risks by ticker (simulate load_golden_dataset)\n")

tickers = []
for ticker_name in df['ticker'].unique():
    ticker_data = df[df['ticker'] == ticker_name].sort_values('risk_rank')
    expert_risks = ticker_data['risk_description'].tolist()
    
    tickers.append({
        "ticker": ticker_name,
        "company_name": ticker_data['company_name'].iloc[0],
        "sector": ticker_data['sector'].iloc[0],
        "expert_risks": expert_risks
    })

print(f"Grouped into {len(tickers)} ticker entries")
print(f"\nFirst ticker: {tickers[0]['ticker']} ({tickers[0]['company_name']})")
print(f"  Sector: {tickers[0]['sector']}")
print(f"  Expert risks: {len(tickers[0]['expert_risks'])}")
print(f"\n✓ PASS")

Test 2.2: Group risks by ticker (simulate load_golden_dataset)

Grouped into 10 ticker entries

First ticker: PLD (Prologis)
  Sector: Industrial/Logistics
  Expert risks: 5

✓ PASS


In [16]:
# Test 2.3: Build sector-risks mapping
print("Test 2.3: Build all_sectors_risks mapping\n")

all_sectors_risks = {}
for sector in df['sector'].unique():
    sector_risks = df[df['sector'] == sector]['risk_description'].tolist()
    all_sectors_risks[sector] = sector_risks

print(f"Loaded {len(all_sectors_risks)} sectors:")
for sector, risks in all_sectors_risks.items():
    print(f"  - {sector}: {len(risks)} risks")

print(f"\n✓ PASS")

Test 2.3: Build all_sectors_risks mapping

Loaded 8 sectors:
  - Industrial/Logistics: 5 risks
  - Infrastructure/Towers: 5 risks
  - Data Centers: 10 risks
  - Self Storage: 5 risks
  - Retail (Net Lease): 5 risks
  - Residential/Multifamily: 5 risks
  - Healthcare: 10 risks
  - Retail/Malls: 5 risks

✓ PASS


## Test 3: Evaluation Results

In [18]:
# Test 3.1: Load evaluation results JSON
print("Test 3.1: Load evaluation results\n")

results_path = project_root / "evaluation" / "results" / "evaluation_results.json"

if results_path.exists():
    with open(results_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"Metadata:")
    print(f"  Run date: {results['metadata']['run_date']}")
    print(f"  Total tickers: {results['metadata']['total_tickers']}")
    print(f"  Cached only: {results['metadata'].get('cached_only_mode', False)}")
    print(f"\nTickers processed: {len(results['tickers'])}")
    print(f"\n✓ PASS")
else:
    print(f"⚠ No results file found at {results_path}")
    print("Run: python -m evaluation.run_evaluation --cached-only")

Test 3.1: Load evaluation results

Metadata:
  Run date: 2026-01-23T03:03:55.919099
  Total tickers: 10
  Cached only: True

Tickers processed: 3

✓ PASS


In [20]:
# Test 3.2: Examine ticker results
print("Test 3.2: Examine individual ticker results\n")

if results_path.exists():
    for ticker_result in results['tickers'][:3]:  # First 3 tickers
        print(f"Ticker: {ticker_result['ticker']} ({ticker_result['company_name']})")
        print(f"  Sector: {ticker_result['sector']}")
        print(f"  Source: {ticker_result['source']}")
        print(f"  Generated risks: {len(ticker_result['risks'])}")
        print(f"  Expert risks: {len(ticker_result['expert_risks'])}")
        
        if 'metrics' in ticker_result:
            metrics = ticker_result['metrics']
            print(f"  Metrics:")
            print(f"    - Semantic Similarity: {metrics['semantic_similarity']:.3f}")
            print(f"    - NDCG@5: {metrics['ndcg_at_5']:.3f}")
            print(f"    - Sector-Specificity: {metrics['sector_specificity']:.3f}")
        print()
    
    print(f"✓ PASS")
else:
    print("⚠ No results file to examine")

Test 3.2: Examine individual ticker results

Ticker: AMT (American Tower)
  Sector: Infrastructure/Towers
  Source: cached_golden_output
  Generated risks: 5
  Expert risks: 5
  Metrics:
    - Semantic Similarity: 1.000
    - NDCG@5: 1.000
    - Sector-Specificity: 0.728

Ticker: EQIX (Equinix)
  Sector: Data Centers
  Source: cached_golden_output
  Generated risks: 5
  Expert risks: 5
  Metrics:
    - Semantic Similarity: 0.551
    - NDCG@5: 0.793
    - Sector-Specificity: 0.540

Ticker: PLD (Prologis)
  Sector: Industrial/Logistics
  Source: cached_golden_output
  Generated risks: 5
  Expert risks: 5
  Metrics:
    - Semantic Similarity: 0.703
    - NDCG@5: 0.908
    - Sector-Specificity: 0.593

✓ PASS


In [21]:
# Test 3.3: Verify aggregate metrics
print("Test 3.3: Verify aggregate metrics calculation\n")

if results_path.exists() and 'summary_metrics' in results:
    summary = results['summary_metrics']
    
    print(f"Aggregate Metrics (n={len(results['tickers'])}):\n")
    
    for metric_name, stats in summary.items():
        print(f"{metric_name}:")
        print(f"  Mean: {stats['mean']:.3f}")
        print(f"  Min:  {stats['min']:.3f}")
        print(f"  Max:  {stats['max']:.3f}")
        print()
    
    print(f"✓ PASS")
else:
    print("⚠ No summary metrics found")

Test 3.3: Verify aggregate metrics calculation

Aggregate Metrics (n=3):

semantic_similarity:
  Mean: 0.751
  Min:  0.551
  Max:  1.000

ndcg_at_5:
  Mean: 0.900
  Min:  0.793
  Max:  1.000

sector_specificity:
  Mean: 0.621
  Min:  0.540
  Max:  0.728

✓ PASS


## Test 4: Mock Data Quality Levels

In [22]:
# Test 4.1: Verify mock data exists
print("Test 4.1: Check mock LLM outputs\n")

mock_tickers = ['AMT', 'PLD', 'EQIX']
mock_files = []

for ticker in mock_tickers:
    has_cache = golden_manager.has_cached_output(ticker)
    print(f"{ticker}: {'✓ Cached' if has_cache else '✗ Not cached'}")
    if has_cache:
        mock_files.append(ticker)

print(f"\nMock files found: {len(mock_files)}/3")
print(f"✓ PASS" if len(mock_files) == 3 else "⚠ Some mock files missing")

Test 4.1: Check mock LLM outputs

AMT: ✓ Cached
PLD: ✓ Cached
EQIX: ✓ Cached

Mock files found: 3/3
✓ PASS


In [23]:
# Test 4.2: Analyze quality levels from results
print("Test 4.2: Analyze mock data quality levels\n")

if results_path.exists():
    quality_analysis = []
    
    for ticker in mock_tickers:
        ticker_data = next((t for t in results['tickers'] if t['ticker'] == ticker), None)
        if ticker_data and 'metrics' in ticker_data:
            m = ticker_data['metrics']
            quality_analysis.append({
                'ticker': ticker,
                'similarity': m['semantic_similarity'],
                'ndcg': m['ndcg_at_5'],
                'specificity': m['sector_specificity']
            })
    
    if quality_analysis:
        df_quality = pd.DataFrame(quality_analysis)
        print(df_quality.to_string(index=False))
        
        print("\nQuality Level Assessment:")
        for item in quality_analysis:
            if item['similarity'] > 0.95:
                level = "Perfect Match"
            elif item['similarity'] > 0.70:
                level = "Good Quality"
            else:
                level = "Mediocre Quality"
            print(f"  {item['ticker']}: {level} (similarity={item['similarity']:.3f})")
        
        print(f"\n✓ PASS")
    else:
        print("⚠ No mock data metrics found")
else:
    print("⚠ No results file to analyze")

Test 4.2: Analyze mock data quality levels

ticker  similarity     ndcg  specificity
   AMT    1.000000 1.000000     0.728184
   PLD    0.702755 0.908284     0.593391
  EQIX    0.551109 0.793001     0.540010

Quality Level Assessment:
  AMT: Perfect Match (similarity=1.000)
  PLD: Good Quality (similarity=0.703)
  EQIX: Mediocre Quality (similarity=0.551)

✓ PASS


## Test 5: End-to-End Pipeline

In [24]:
# Test 5.1: Simulate evaluation for single ticker
print("Test 5.1: Simulate single ticker evaluation\n")

from evaluation.metrics import evaluate_summary

if len(mock_files) > 0:
    test_ticker = mock_files[0]
    
    # Load cached output
    cached_summary = golden_manager.load_cached_output(test_ticker)
    
    # Get expert risks
    ticker_info = next((t for t in tickers if t['ticker'] == test_ticker), None)
    
    if cached_summary and ticker_info:
        print(f"Evaluating {test_ticker}...")
        
        # Calculate metrics
        metrics = evaluate_summary(
            generated_risks=cached_summary.risks,
            golden_risks=ticker_info['expert_risks'],
            sector=ticker_info['sector'],
            all_sectors_risks=all_sectors_risks
        )
        
        print(f"\nResults:")
        print(f"  Semantic Similarity: {metrics['semantic_similarity']:.4f}")
        print(f"  NDCG@5: {metrics['ndcg_at_5']:.4f}")
        print(f"  Sector-Specificity: {metrics['sector_specificity']:.4f}")
        print(f"\n✓ PASS")
    else:
        print("✗ FAIL: Could not load data")
else:
    print("⚠ No mock files to test")

Test 5.1: Simulate single ticker evaluation

Evaluating AMT...

Results:
  Semantic Similarity: 1.0000
  NDCG@5: 1.0000
  Sector-Specificity: 0.7282

✓ PASS


In [25]:
# Test 5.2: Verify metrics meet targets
print("Test 5.2: Check if metrics meet target thresholds\n")

if results_path.exists():
    targets = {
        'semantic_similarity': 0.75,
        'ndcg_at_5': 0.70,
        'sector_specificity': 0.40
    }
    
    ticker_passes = 0
    total_tickers = len(results['tickers'])
    
    for ticker_data in results['tickers']:
        if 'metrics' not in ticker_data:
            continue
            
        metrics = ticker_data['metrics']
        meets_targets = all(
            metrics[key] >= threshold 
            for key, threshold in targets.items()
        )
        
        status = "✓" if meets_targets else "⚠"
        print(f"{status} {ticker_data['ticker']}: "
              f"sim={metrics['semantic_similarity']:.3f}, "
              f"ndcg={metrics['ndcg_at_5']:.3f}, "
              f"spec={metrics['sector_specificity']:.3f}")
        
        if meets_targets:
            ticker_passes += 1
    
    print(f"\n{ticker_passes}/{total_tickers} tickers meet all target thresholds")
    print(f"\n✓ PASS (validation complete)")
else:
    print("⚠ No results file to check")

Test 5.2: Check if metrics meet target thresholds

✓ AMT: sim=1.000, ndcg=1.000, spec=0.728
⚠ EQIX: sim=0.551, ndcg=0.793, spec=0.540
⚠ PLD: sim=0.703, ndcg=0.908, spec=0.593

1/3 tickers meet all target thresholds

✓ PASS (validation complete)


## Test 6: Command-Line Simulation

In [26]:
# Test 6.1: Display how to run evaluation with different modes
print("Test 6.1: Command-line usage examples\n")

commands = [
    ("Use cached outputs only", "python -m evaluation.run_evaluation --cached-only"),
    ("Regenerate all outputs", "python -m evaluation.run_evaluation --regenerate"),
    ("Process specific tickers", "python -m evaluation.run_evaluation --tickers AMT PLD EQIX"),
    ("Normal mode (use cache if available)", "python -m evaluation.run_evaluation"),
]

for description, command in commands:
    print(f"{description}:")
    print(f"  {command}\n")

print("✓ Reference commands displayed")

Test 6.1: Command-line usage examples

Use cached outputs only:
  python -m evaluation.run_evaluation --cached-only

Regenerate all outputs:
  python -m evaluation.run_evaluation --regenerate

Process specific tickers:
  python -m evaluation.run_evaluation --tickers AMT PLD EQIX

Normal mode (use cache if available):
  python -m evaluation.run_evaluation

✓ Reference commands displayed


## Summary

### Evaluation Runner Components

✅ **GoldenOutputManager**
- Caches LLM outputs as JSON files
- Loads cached outputs for evaluation
- Lists available cached tickers

✅ **Golden Dataset Loading**
- Loads expert-curated risks from CSV
- Groups risks by ticker
- Builds sector-specific risk mappings

✅ **Evaluation Pipeline**
- Processes tickers individually
- Calculates all Phase 2 metrics
- Aggregates results across tickers
- Saves comprehensive JSON results

✅ **Mock Data Strategy**
- 3 quality levels (perfect, good, mediocre)
- Validates metrics discrimination
- Enables testing without Groq API

### Files Created
- `evaluation/golden_outputs/*.json` - Cached LLM outputs
- `evaluation/results/evaluation_results.json` - Evaluation metrics

### Next Steps
- Run full evaluation when Groq tokens available
- Generate visualization/reporting dashboards
- Compare different LLM models or prompts